[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c12_laws.ipynb)

# The planet found with a pen: differential equations

On 23 September 1846 Johann Galle found Neptune within an hour, where Urbain Le Verrier's letter told him to look.
Le Verrier found it with a rule for change. Here the same idea runs on a cup of tea: a rule for how fast it cools,
plus a start, gives the whole future.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, ToggleButtons

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the curves · the rates (slope marks) · the integrals

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

ROOM = 20.0                                  # the room, °C
K = math.log(4 / 3) / 5                      # this cup: from 80 °C to 65 °C in five minutes

def follow(rule, T, h, minutes):
    """Follow a rule for change in steps of h minutes: work out the rate, take a step, repeat."""
    ts, Ts = [0.0], [T]
    for _ in range(int(round(minutes / h))):
        T = T + rule(T) * h
        ts.append(ts[-1] + h); Ts.append(T)
    return np.array(ts), np.array(Ts)

print("Ready. Run the next cell.")

## 1 · A field of slopes: the rule, drawn everywhere

Newton's law of cooling: **dT/dt = −k(T − Tₛ)**, "the rate the temperature changes is minus k times how much hotter
the tea is than the room". At every time and temperature, the rule says how steeply the temperature falls, so we can
draw a little slope mark there. Click a start, and a curve flows from it, following the marks. Drag k to change how
fast this cup lets heat go.

In [ ]:
def slope_field(start="80 °C", k=0.06):
    T0 = float(start.split()[0])
    rule = lambda T: -k * (T - ROOM)
    fig, ax = plt.subplots(figsize=(10, 5.6))
    for t in np.linspace(0, 40, 21):
        for T in np.linspace(5, 100, 20):
            m = rule(T); L = 0.8
            dt = L / math.hypot(1, m / 3.0)
            ax.plot([t - dt, t + dt], [T - m * dt, T + m * dt], color=SAGE, lw=1.1, alpha=0.7)
    ts, Ts = follow(rule, T0, 0.05, 40)
    ax.plot(ts, Ts, color=CORAL, lw=3, label=f"start {T0:.0f} °C")
    ax.plot([0], [T0], "o", color=CORAL, ms=10)
    ax.axhline(ROOM, color=DIM, ls="--", lw=1.2, label="the room, 20 °C")
    ax.set_xlim(0, 40); ax.set_ylim(0, 105); ax.set_xlabel("minutes"); ax.set_ylabel("temperature (°C)")
    ax.set_title(f"dT/dt = −{k:.2f} (T − 20):  from {T0:.0f} °C, {Ts[100]:.1f} °C after 5 minutes")
    ax.legend(loc="upper right"); plt.show()

interact(slope_field, start=ToggleButtons(options=["95 °C", "80 °C", "50 °C", "5 °C"], value="80 °C", description="start"),
         k=FloatSlider(value=0.06, min=0.02, max=0.2, step=0.01, description="k"));

**Notice.** Every start flows along its own curve, and every curve heads for the room's 20 °C: the whole family is the
**general solution**, and the start picks one, the **particular solution**. A start below the room warms up.

## 2 · The steps land on the formula

Follow the rule a minute at a time from 80 °C, then half a minute, then a tenth. Separating the variables takes all
the steps at once: **T = 20 + 60e^(−kt)**. The smaller the steps, the closer they land on it.

In [ ]:
t = np.linspace(0, 30, 300)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, ROOM + 60 * np.exp(-K * t), color=TEAL, lw=6, alpha=0.5, label="T = 20 + 60e^(−kt)")
for h, col in ((1.0, DIM), (0.5, SAGE), (0.1, CORAL)):
    ts, Ts = follow(lambda T: -K * (T - ROOM), 80.0, h, 30)
    ax.plot(ts, Ts, "o-" if h >= 0.5 else "-", color=col, ms=4, lw=1.4, label=f"steps of {h:g} min")
    print(f"steps of {h:>3g} min: {Ts[int(round(5 / h))]:.2f} °C at 5 minutes")
print(f"the formula:     {ROOM + 60 * math.exp(-K * 5):.2f} °C at 5 minutes")
ax.set_xlabel("minutes"); ax.set_ylabel("temperature (°C)"); ax.legend(loc="upper right")
ax.set_title("rule + start: the steps close in on one curve"); plt.show()

**Notice.** One-minute steps reach 64.6 °C at five minutes, half-minute steps 64.8, tenth-of-a-minute steps 64.96.
The formula says 65.

## 3 · When can you drink it?

The gap above the room shrinks by a quarter every five minutes: 60 → 45 → 33.75 → … Find when the tea reaches a
temperature you choose.

In [ ]:
def when(target=60.0):
    t = 5 * math.log(60 / (target - ROOM)) / math.log(4 / 3)
    print(f"{target:.0f} °C after {t:.2f} minutes")

interact(when, target=FloatSlider(value=60, min=25, max=75, step=5, description="target °C"));

## 4 · Check your answer (sympy)

The cooling law, the homogeneous equation dy/dx = y/x + x/y with y(1) = 2, and the linear equation dy/dx + y = x
with y(0) = 1, solved by the computer.

In [ ]:
import sympy as sp
tt, x, k = sp.symbols("t x k", positive=True)
T, y = sp.Function("T"), sp.Function("y")
print(sp.dsolve(sp.Eq(T(tt).diff(tt), -k * (T(tt) - 20)), T(tt), ics={T(0): 80}))
print(sp.dsolve(sp.Eq(y(x).diff(x) + y(x), x), y(x), ics={y(0): 1}))
hy = x * sp.sqrt(2 * sp.log(x) + 4)
print("y = x√(2 ln x + 4) fits dy/dx = y/x + x/y:", sp.simplify(hy.diff(x) - (hy / x + x / hy)) == 0, " y(1) =", hy.subs(x, 1))

## Sources
J. R. Lévy, "Le Verrier", *Dictionary of Scientific Biography*; APS News, "Neptune's existence confirmed"; Newton's
"Scala graduum caloris", *Philosophical Transactions* (1701); nobelprize.org, Chemistry 1960. CBSE Mathematics
syllabus 2026–27 (Differential Equations). All retrieved 2026-10-24.